# SBA Modeling Notebook

This notebook implements the notebook-first modeling workflow for the SBA migration project.

The source data is the canonical featurized export produced by `notebooks/sba_featurization.ipynb`. We do not create a second random split here. The train/validation/test partitions come from the `dataset_partition` column in the model-ready data.

The first-stage model is calibrated on training data. Its operating cutoff is selected on validation by finding the elbow of the cumulative gains curve, and that cutoff is frozen before the test set is evaluated.


In [ ]:
from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.pipeline import Pipeline

working_directory = Path.cwd().resolve()
project_root_candidates = [working_directory, *working_directory.parents]
PROJECT_ROOT = next(
    (
        candidate
        for candidate in project_root_candidates
        if (candidate / 'data' / 'featurization' / 'model_ready_numeric_data.csv').exists()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f'Could not locate the project root from current working directory: {working_directory}'
    )

MODEL_READY_PATH = PROJECT_ROOT / 'data' / 'featurization' / 'model_ready_numeric_data.csv'
ARTIFACT_DIR = PROJECT_ROOT / 'data' / 'modeling'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
TARGET_COL = 'loan_status_r'
PARTITION_COL = 'dataset_partition'

model_ready_df = pd.read_csv(MODEL_READY_PATH)
print(f'Read {len(model_ready_df)} rows from {MODEL_READY_PATH}')
print(model_ready_df[PARTITION_COL].value_counts().to_dict())
assert TARGET_COL in model_ready_df.columns, f'Missing target column: {TARGET_COL}'
assert PARTITION_COL in model_ready_df.columns, f'Missing partition column: {PARTITION_COL}'
feature_cols = [
    column for column in model_ready_df.columns
    if column not in {TARGET_COL, PARTITION_COL}
]
print(f'Using {len(feature_cols)} feature columns')

def prepare_partitions(frame: pd.DataFrame):
    partitions = {}
    for name in ['train', 'validation', 'test']:
        subset = frame[frame[PARTITION_COL] == name].copy()
        if subset.empty:
            raise ValueError(f'{name} partition is empty in the model-ready data.')
        subset = subset[subset[TARGET_COL].isin([0, 1])].copy().reset_index(drop=True)
        partitions[name] = subset
    return partitions['train'], partitions['validation'], partitions['test']

train_df, validation_df, test_df = prepare_partitions(model_ready_df)
print('Partitions:', train_df.shape, validation_df.shape, test_df.shape)

X_train = train_df[feature_cols].copy()
y_train = train_df[TARGET_COL].astype(int).to_numpy()
X_val = validation_df[feature_cols].copy()
y_val = validation_df[TARGET_COL].astype(int).to_numpy()
X_test = test_df[feature_cols].copy()
y_test = test_df[TARGET_COL].astype(int).to_numpy()


In [3]:
candidate_models = {
    'gradient_boosting': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model', GradientBoostingClassifier(random_state=42)),
    ]),
    'random_forest': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model', RandomForestClassifier(
            n_estimators=400,
            max_depth=None,
            random_state=42,
            class_weight='balanced_subsample',
        )),
    ]),
}

validation_scores = []
calibrators = {}
for name, model in candidate_models.items():
    calibrator = CalibratedClassifierCV(
        estimator=model,
        method='isotonic',
        cv=3,
    )
    calibrator.fit(X_train, y_train)
    val_prob = calibrator.predict_proba(X_val)[:, 1]
    pr_auc = average_precision_score(y_val, val_prob)
    validation_scores.append({'model_name': name, 'validation_pr_auc': float(pr_auc)})
    calibrators[name] = calibrator
    print(f'{name}: validation PR-AUC = {pr_auc:.4f}')

validation_summary = pd.DataFrame(validation_scores).sort_values('validation_pr_auc', ascending=False)
champion_name = validation_summary.iloc[0]['model_name']
champion_model = candidate_models[champion_name]
champion_calibrator = calibrators[champion_name]
print(f'Champion model selected: {champion_name}')


gradient_boosting: validation PR-AUC = 0.1560
random_forest: validation PR-AUC = 0.1673
Champion model selected: random_forest


In [4]:
def choose_recall_maximizing_threshold(scores, actual):
    score_table = pd.DataFrame({'score': scores, 'actual': actual})
    unique_scores = np.unique(np.sort(score_table['score'].to_numpy()))
    best_threshold = None
    best_recall = -1.0
    best_precision = -1.0

    for threshold in unique_scores:
        predicted_positive = (score_table['score'] >= threshold).astype(int)
        true_positive = int(((predicted_positive == 1) & (score_table['actual'] == 1)).sum())
        actual_positive = int((score_table['actual'] == 1).sum())
        recall = true_positive / actual_positive if actual_positive else 0.0
        precision = true_positive / predicted_positive.sum() if predicted_positive.sum() else 0.0

        if recall > best_recall + 1e-12 or (abs(recall - best_recall) <= 1e-12 and precision > best_precision + 1e-12):
            best_threshold = float(threshold)
            best_recall = recall
            best_precision = precision

    return best_threshold, best_recall, best_precision

val_prob = champion_calibrator.predict_proba(X_val)[:, 1]
first_stage_threshold, max_validation_recall, max_validation_precision = choose_recall_maximizing_threshold(val_prob, y_val)
print(f'First-stage recall-maximizing threshold on validation: {first_stage_threshold:.6f}')
print(f'Max validation recall: {max_validation_recall:.4f}; precision at that point: {max_validation_precision:.4f}')


First-stage recall-maximizing threshold on validation: 0.000000
Max validation recall: 1.0000; precision at that point: 0.0918


In [ ]:
def build_cumulative_gains_table(scores, actual, bins=10):
    ranked = pd.DataFrame({'score': scores, 'actual': actual}).sort_values(
        'score', ascending=False, kind='mergesort'
    ).reset_index(drop=True)
    total_defaults = int(ranked['actual'].sum())
    if total_defaults == 0:
        raise ValueError('Cannot build a cumulative gains curve without validation defaults.')

    cutoff_counts = np.ceil(
        np.arange(1, bins + 1) * len(ranked) / bins
    ).astype(int)
    records = []
    for cutoff_count in cutoff_counts:
        reviewed = ranked.iloc[:cutoff_count]
        population_fraction = cutoff_count / len(ranked)
        defaults_captured = int(reviewed['actual'].sum())
        cumulative_gains = defaults_captured / total_defaults
        records.append({
            'review_percent': population_fraction * 100.0,
            'review_count': cutoff_count,
            'population_fraction': population_fraction,
            'defaults_captured': defaults_captured,
            'cumulative_gains': cumulative_gains,
            'lift': cumulative_gains / population_fraction,
            'score_cutoff': float(reviewed['score'].min()),
            'precision_at_cutoff': float(reviewed['actual'].mean()),
        })
    return pd.DataFrame(records)


def select_elbow_cutoff(gains_table):
    distance_from_random = (
        gains_table['cumulative_gains'] - gains_table['population_fraction']
    )
    positive_distance = distance_from_random > 1e-12
    if not positive_distance.any():
        return gains_table.iloc[-1].copy(), 0.0

    candidates = gains_table.loc[positive_distance].copy()
    candidates['_elbow_distance'] = distance_from_random.loc[positive_distance]
    max_distance = candidates['_elbow_distance'].max()
    selected = candidates.loc[
        np.isclose(candidates['_elbow_distance'], max_distance, rtol=0.0, atol=1e-12)
    ].sort_values('review_percent').iloc[0]
    return selected, float(max_distance)


validation_gains_table = build_cumulative_gains_table(val_prob, y_val, bins=10)
selected_elbow, elbow_distance = select_elbow_cutoff(validation_gains_table)
selected_review_percent = float(selected_elbow['review_percent'])
selected_threshold = float(selected_elbow['score_cutoff'])
selected_cumulative_gains = float(selected_elbow['cumulative_gains'])
selected_lift = float(selected_elbow['lift'])
selected_threshold_review_rate = float(np.mean(val_prob >= selected_threshold))

print('Validation cumulative gains and lift table:')
print(validation_gains_table.to_string(index=False))
print(f'Selected elbow review percentile: {selected_review_percent:.1f}%')
print(f'Elbow distance above random baseline: {elbow_distance:.4f}')
print(f'Frozen operating score threshold: {selected_threshold:.6f}')
print(f'Realized validation review rate at score threshold: {selected_threshold_review_rate:.4f}')
if selected_threshold_review_rate > selected_review_percent / 100.0 + 1e-12:
    print(
        'Warning: score ties at the cutoff increase the realized review rate '
        'above the selected batch percentile.'
    )


In [ ]:
val_pred = (val_prob >= selected_threshold).astype(int)
validation_review_rate = float(np.mean(val_pred))
validation_recall = recall_score(y_val, val_pred, zero_division=0)
validation_lift = validation_recall / validation_review_rate if validation_review_rate else 0.0
validation_metrics = {
    'model_name': champion_name,
    'first_stage_threshold': first_stage_threshold,
    'selected_review_percent': selected_review_percent,
    'selected_threshold': selected_threshold,
    'selected_cumulative_gains': selected_cumulative_gains,
    'selected_lift': selected_lift,
    'elbow_distance': elbow_distance,
    'precision': precision_score(y_val, val_pred, zero_division=0),
    'recall': validation_recall,
    'f1': f1_score(y_val, val_pred, zero_division=0),
    'balanced_accuracy': balanced_accuracy_score(y_val, val_pred),
    'pr_auc': average_precision_score(y_val, val_prob),
    'realized_lift': validation_lift,
    'review_rate': validation_review_rate,
    'n_reviewed': int(val_pred.sum()),
    'n_validation_rows': int(len(validation_df)),
}
print('Validation metrics at the frozen elbow-derived threshold:')
print(pd.Series(validation_metrics).to_string())


In [ ]:
test_prob = champion_calibrator.predict_proba(X_test)[:, 1]
test_pred = (test_prob >= selected_threshold).astype(int)
test_review_rate = float(np.mean(test_pred))
test_recall = recall_score(y_test, test_pred, zero_division=0)
test_lift = test_recall / test_review_rate if test_review_rate else 0.0
test_metrics = {
    'model_name': champion_name,
    'selected_review_percent': selected_review_percent,
    'selected_threshold': selected_threshold,
    'precision': precision_score(y_test, test_pred, zero_division=0),
    'recall': test_recall,
    'f1': f1_score(y_test, test_pred, zero_division=0),
    'balanced_accuracy': balanced_accuracy_score(y_test, test_pred),
    'pr_auc': average_precision_score(y_test, test_prob),
    'cumulative_gains': test_recall,
    'lift': test_lift,
    'review_rate': test_review_rate,
    'n_reviewed': int(test_pred.sum()),
    'n_test_rows': int(len(test_df)),
}
print('Final test metrics at the frozen elbow-derived threshold:')
print(pd.Series(test_metrics).to_string())


In [ ]:
model_bundle = {
    'model_name': champion_name,
    'model': champion_model,
    'calibrator': champion_calibrator,
    'first_stage_threshold': first_stage_threshold,
    'threshold_method': 'validation_cumulative_gains_elbow',
    'selected_review_percent': selected_review_percent,
    'selected_threshold': selected_threshold,
    'selected_cumulative_gains': selected_cumulative_gains,
    'selected_lift': selected_lift,
    'elbow_distance': elbow_distance,
    'feature_columns': feature_cols,
    'target_column': TARGET_COL,
    'partition_column': PARTITION_COL,
    'validation_gains_lift_table': validation_gains_table,
    'validation_metrics': validation_metrics,
    'test_metrics': test_metrics,
    'data_path': str(MODEL_READY_PATH),
}
joblib.dump(model_bundle, ARTIFACT_DIR / 'sba_model_bundle.joblib')
validation_gains_table.to_csv(
    ARTIFACT_DIR / 'validation_gains_lift.csv', index=False
)
print('Model bundle saved to', ARTIFACT_DIR / 'sba_model_bundle.joblib')
print('Validation gains/lift table saved to', ARTIFACT_DIR / 'validation_gains_lift.csv')


## Notes

- The model learns only from the canonical `train` partition and is calibrated with isotonic regression using training data only.
- The validation split is scored once to build cumulative gains and lift at ordered review percentiles.
- Stage 2 selects the elbow as the point with the greatest positive distance above the random-selection baseline. If no point is above that baseline, the 100% review cutoff is used conservatively.
- Both the selected review percentile and the calibrated probability at its boundary are frozen before the test evaluation.
- Score ties at the boundary can make the realized `score >= threshold` review population larger than the selected batch percentile; the notebook reports this explicitly.
- The `test` partition is held out for a single final evaluation after the operating cutoff is frozen.
- Raw business-value columns are not required by this gains/lift thresholding workflow.
- This notebook is the canonical notebook-driven workflow; the CLI utility can wrap the same logic later without changing the contract.
